In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import re
from datetime import datetime
from datetime import date
from dateutil.relativedelta import relativedelta

# Importa os dados

In [3]:
df = pd.read_excel(
    "dados.xlsx",
    sheet_name=0,
    header=1,          # usa a 2ª linha como cabeçalho real
    dtype={"Class. Por idade": str},  # evita converter "1" em número se quiser
)


## Criando Pipeline de Limpeza dos Dados

In [6]:
def limpa_dados(df):
    
    colunas_data = ["data_nascimento","data_primeira_consulta","data_primeira_colonoscopia"]

    def padroniza_colunas(df):
        df.columns = (df.columns.astype(str).str.replace("\n", " ", regex=False).str.replace(r"\s+", " ", regex=True).str.strip())
        return df

    def renomeia_colunas(df):
        df = df.rename(columns={
            "Nome": "nome",
            "Class. Por idade": "classificacao_idade",
            "Data nascimento": "data_nascimento",
            "Data 1ª consulta": "data_primeira_consulta",
            "Tempo entre início dos sintomas e primeira consulta": "intervalo_sintomas_primeira_consulta",
            "Sexo": "sexo",
            "Tipo": "tipo",
            "Perda de peso": "perda_peso",
            "Idade (meses) início dos sintomas": "intervalo_nascimento_sintomas",
            "Data 1ª colono": "data_primeira_colonoscopia",
            "Unnamed: 10": "intervalo_sintomas_primeira_colonoscopia",
            "Classificação": "classificacao",
        })
        return df

    def formata_colunas_data(df):
        for coluna in colunas_data:
            df[coluna] = (pd.to_datetime(df[coluna], errors="coerce").dt.strftime("%d/%m/%Y"))
        return df

    def deixar_coluna_inteira(df, coluna):
        for index, tempo in enumerate(df[coluna].tolist()):
            if tempo != '?':
                numero = int(re.sub(r"\D", "", tempo))
                df.loc[index, coluna] = numero
            elif tempo == "?":
                df.loc[index, coluna] = -1
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce")
            df[coluna] = df[coluna].astype("Int64")
        return df

        
    def criar_coluna(df, coluna):
        datas = pd.to_datetime(df['data_nascimento'],format="%d/%m/%Y",errors="coerce")
        meses = df["intervalo_nascimento_sintomas"]
        df[coluna] = [d + pd.DateOffset(months=int(m)) if pd.notna(d) and pd.notna(m) else pd.NaT for d, m in zip(datas, meses)  ]
        df[coluna] = (pd.to_datetime(df[coluna],errors="coerce" ).dt.strftime("%d/%m/%Y") )
        return df


    def calcula_intervalo_colonoscopia(df):
        resultados = []
        for index in range(len(df)):
            d1_str = df.loc[index, "data_inicio_sintomas"]
            d2_str = df.loc[index, "data_primeira_colonoscopia"]
            # 1) converte para datetime
            try:
                d1 = datetime.strptime(d1_str, "%d/%m/%Y")
                d2 = datetime.strptime(d2_str, "%d/%m/%Y")
            except (ValueError, TypeError):
                resultados.append(None)
                continue
            # 2) valida ordem
            if d2 <= d1:
                resultados.append(None)
                continue
            # 3) calcula meses completos
            meses = ((d2.year - d1.year) * 12 + (d2.month - d1.month) - (d2.day < d1.day))

            resultados.append(meses)

        df["intervalo_sintomas_primeira_colonoscopia_recalculado"] = resultados
        df["intervalo_sintomas_primeira_colonoscopia_recalculado"] = pd.to_numeric(df["intervalo_sintomas_primeira_colonoscopia_recalculado"],errors="coerce")
        df["intervalo_sintomas_primeira_colonoscopia_recalculado"] = df["intervalo_sintomas_primeira_colonoscopia_recalculado"].astype("Int64")

        return df


    df = df.drop(index=0).reset_index(drop=True)
    df = padroniza_colunas(df)
    df = renomeia_colunas(df)
    df = formata_colunas_data(df)
    df = df.drop(columns=["nome"])
    df = deixar_coluna_inteira(df,"intervalo_sintomas_primeira_consulta")
    df = deixar_coluna_inteira(df,"intervalo_sintomas_primeira_colonoscopia")
    df = criar_coluna(df,"data_inicio_sintomas")  
    df = calcula_intervalo_colonoscopia(df)
   

    return df

In [8]:
new_df = limpa_dados(df)

In [42]:
#organizar a disposicao das colunas
new_df[
[
"data_nascimento",



"intervalo_nascimento_sintomas",
    
"data_inicio_sintomas",
    
"intervalo_sintomas_primeira_consulta",
    
"data_primeira_consulta",  

"intervalo_sintomas_primeira_colonoscopia_recalculado",

"intervalo_sintomas_primeira_colonoscopia",    
    
"data_primeira_colonoscopia",
    
# "classificacao_idade"
]
]

,data_nascimento,intervalo_nascimento_sintomas,data_inicio_sintomas,intervalo_sintomas_primeira_consulta,data_primeira_consulta,intervalo_sintomas_primeira_colonoscopia_recalculado,intervalo_sintomas_primeira_colonoscopia,data_primeira_colonoscopia
0,27/09/2013,60,27/09/2018,54,02/03/2023,9,6,15/07/2019
1,25/10/2024,14,25/12/2025,5,09/06/2026,4,4,20/05/2026
2,01/01/2006,168,01/01/2020,13,04/05/2021,3,1,13/04/2020
3,19/04/2019,18,19/10/2020,34,03/08/2023,18,19,01/05/2022
4,18/05/2012,108,18/05/2021,20,19/01/2023,20,30,26/01/2023
...,...,...,...,...,...,...,...,...
96,08/08/2012,96,08/08/2020,26,20/10/2022,24,25,01/09/2022
97,05/03/2004,84,05/03/2011,4,19/07/2011,2,3,01/06/2011
98,25/04/2008,144,25/04/2020,3,05/01/2021,6,1,26/10/2020
99,19/05/2017,47,19/04/2021,11,11/03/2022,5,5,29/09/2021


In [12]:
new_df.to_csv('new_df.csv', index=False)